# QT - Quantum Turbulence in a Dipolar Supersolid

Validated eGPE split-step solver.  **`Runtime -> Run all`.**  Nothing to edit.

Outputs go to `MyDrive/Research/Quantum_Turbulence/<date>_<time>_<title>/`
with `checkpoints/ diagnostics/ figures/ logs/`.  Run time 8-12 minutes.

**Version 1.1** fixes a temporal-aliasing blow-up present in v1.0:
the demo used a timestep at the aliasing resonance and diverged near
t=35 with the norm still conserved.  The solver now rejects such
timesteps, and five regression checks (T7) pin the fix down.

## 1. Setup - clone, import, mount Drive

In [ ]:
# CELL 1 - Setup.  Run this first.  Nothing to edit.
import os, sys, subprocess

REPO = 'https://github.com/vk001king/qt.git'
DEST = '/content/qt'

if os.path.isdir(DEST + '/.git'):
    print('Repo present; pulling latest...')
    subprocess.run(['git', '-C', DEST, 'pull', '-q'], check=False)
else:
    print('Cloning', REPO)
    r = subprocess.run(['git', 'clone', '-q', REPO, DEST],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone failed:\n' + r.stderr)
print('Repo ready at', DEST)

need = ['__init__.py', 'kernels.py', 'lhy.py', 'solver.py',
        'diagnostics.py', 'drive_io.py']
have = os.listdir(DEST + '/qtsim')
miss = [f for f in need if f not in have]
if miss:
    raise RuntimeError('Repo incomplete, missing: %s' % miss)
print('All 6 package files present')

if DEST in sys.path:
    sys.path.remove(DEST)
sys.path.insert(0, DEST)
for m in [k for k in sys.modules if k.split('.')[0] == 'qtsim']:
    del sys.modules[m]

import qtsim
from qtsim import Grid, EGPESolver, Archive
assert qtsim.selfcheck(), 'qtsim self-check failed'
print('Loaded qtsim v%s from %s'
      % (qtsim.__version__, os.path.dirname(qtsim.__file__)))

archive = Archive(base='Research', project='Quantum_Turbulence')
archive.list_runs()

## 2. Validation - 23 checks against analytic results

If anything fails, stop: nothing below can be trusted.

In [ ]:
# CELL 2 - Validation suite (23 checks).  About 60 s.
import subprocess, sys

run = archive.new_run('validation suite',
                      params={'version': qtsim.__version__})
r = subprocess.run([sys.executable, 'tests/run_validation.py'],
                   cwd=DEST, capture_output=True, text=True)
print(r.stdout[-4000:])
if r.stderr.strip():
    print('STDERR:', r.stderr[-1500:])

run.save_text(r.stdout, 'validation-output')
if r.returncode == 0:
    run.finish('completed')
    print('RESULT: all checks passed.')
else:
    run.finish('failed')
    print('RESULT: failures above.  Do not trust the physics below.')

## 3. Vortex pair - motion and sound emission

Energy drift is printed alongside the physics as a live health check.

In [ ]:
# CELL 3 - Vortex pair: detection, motion, sound emission.
#
# TIMESTEP NOTE (this bit matters).  The kinetic substep applies
# exp(-i k^2 dt) exactly, so there is no CFL stability limit -- but the
# highest grid mode advances k_max^2 * dt radians per step.  Once that
# approaches pi those modes are unresolved in time and the nonlinear
# term pumps them until the field detonates, WITH THE NORM STILL
# CONSERVED to round-off.  An earlier version of this notebook used
# 256^2 with dt=0.02 (6.3 rad/step) and blew up near t=35: two vortices
# became 21000.  The solver now refuses such timesteps outright.
import numpy as np, matplotlib.pyplot as plt
from qtsim import Grid, EGPESolver
from qtsim import plaquette_charges_2d, helmholtz_split_2d

N, L, dt = 128, 64.0, 0.01
grid = Grid((N, N), (L, L))
dx = grid.dx[0]

run = archive.new_run('vortex pair dynamics',
                      params={'N': N, 'L': L, 'dt': dt})

probe = EGPESolver(grid)
print('grid  dx = %.3f xi   (need dx <= 0.5 to resolve the core)' % dx)
print('phase advance of highest mode = %.3f rad/step'
      % probe.max_phase_per_step(dt))
print('largest safe dt on this grid  = %.4f' % probe.suggested_dt())
print()

# half-cell offsets: a core sitting exactly on a grid point has
# undefined phase there and corrupts the winding detector
xp, xm, yc = 8.0 + 0.5*dx, -8.0 + 0.5*dx, 0.5*dx
thp = np.arctan2(grid.X[1] - yc, grid.X[0] - xp)
thm = np.arctan2(grid.X[1] - yc, grid.X[0] - xm)
rp = np.sqrt((grid.X[0] - xp)**2 + (grid.X[1] - yc)**2)
rm = np.sqrt((grid.X[0] - xm)**2 + (grid.X[1] - yc)**2)

s = EGPESolver(grid)
s.psi = np.tanh(rp) * np.tanh(rm) * np.exp(1j*(thp - thm))
s.step_imag(5e-3, 300, norm_target=s.norm())

q = plaquette_charges_2d(s.psi)
print('vortices detected:  +1 = %d   -1 = %d'
      % (int((q == 1).sum()), int((q == -1).sum())))

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(np.abs(s.psi).T**2, origin='lower',
             extent=[-L/2, L/2, -L/2, L/2], cmap='inferno')
ax[0].set_title('density |psi|^2')
ax[0].set_xlabel('x / xi'); ax[0].set_ylabel('y / xi')
ax[1].imshow(np.angle(s.psi).T, origin='lower',
             extent=[-L/2, L/2, -L/2, L/2], cmap='twilight')
for v in np.argwhere(q == 1):
    ax[1].plot(grid.X[0][v[0], v[1]], grid.X[1][v[0], v[1]],
               'r+', ms=14, mew=2)
for v in np.argwhere(q == -1):
    ax[1].plot(grid.X[0][v[0], v[1]], grid.X[1][v[0], v[1]],
               'bx', ms=14, mew=2)
ax[1].set_title('phase  (red +1, blue -1)')
ax[1].set_xlabel('x / xi')
plt.tight_layout()
run.save_figure(fig, 'pair-initial')
plt.show()

# evolve, watching conservation as a live health check
E0, N0 = s.energy()['total'], s.norm()
ts, Ei_s, Ec_s, nv_s, dE_s = [], [], [], [], []
nsteps = int(100.0 / dt)
print('evolving to t = 100 ...')
print('     t     nv        Ei        Ec      dE/E')
for k in range(nsteps + 1):
    if k % 250 == 0:
        Ei, Ec, _ = helmholtz_split_2d(s.psi, grid)
        nv = int((plaquette_charges_2d(s.psi) != 0).sum())
        dE = abs(s.energy()['total'] - E0) / abs(E0)
        ts.append(k*dt); Ei_s.append(Ei); Ec_s.append(Ec)
        nv_s.append(nv); dE_s.append(dE)
        if k % 2500 == 0:
            print('  %6.1f  %4d  %8.2f  %8.2f  %8.1e'
                  % (k*dt, nv, Ei, Ec, dE))
    if k < nsteps:
        s.step_real(dt)

dE_max = max(dE_s)
print()
print('max energy drift  = %.2e   (healthy if < 1e-5)' % dE_max)
print('norm drift        = %.2e' % (abs(s.norm() - N0)/N0))
print('final vortex count= %d   (started at 2)' % nv_s[-1])
if dE_max > 1e-5 or nv_s[-1] > 4:
    print('WARNING: run is not trustworthy -- treat output as a bug.')
else:
    print('Run is conservative and stable.')

fig2, (a1, a2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
a1.plot(ts, Ei_s, 'b-', label='incompressible (vortex)')
a1.plot(ts, Ec_s, 'r-', label='compressible (sound)')
a1.set_ylabel('energy'); a1.legend()
a1.set_title('energy channels for a travelling vortex pair')
a2.plot(ts, nv_s, 'ko-', ms=3)
a2.set_xlabel('time / tau'); a2.set_ylabel('vortex count')
a2.set_ylim(0, max(6, max(nv_s) + 1))
plt.tight_layout()
run.save_figure(fig2, 'energy-channels')
plt.show()

run.save_diagnostics([{'t': a, 'Ei': b, 'Ec': c, 'nv': d, 'dE_rel': e}
                      for a, b, c, d, e in
                      zip(ts, Ei_s, Ec_s, nv_s, dE_s)])
run.save_checkpoint(s.psi, step=nsteps, t=100.0)
run.finish('completed')

## 4. Dipolar ground state - with an honest modulation test

Reports contrast *and* wavelength, and refuses to call a box-scale
stripe a crystal.

In [ ]:
# CELL 4 - Dipolar ground state, with an HONEST modulation test.
#
# Contrast alone does not prove supersolidity: the largest mode a
# periodic box allows has wavelength = box size, and a single stripe
# filling the box gives a big contrast while being a box artifact.
# So we measure the dominant wavelength too and refuse to call
# anything a crystal unless several periods fit inside the box.
import numpy as np, matplotlib.pyplot as plt
from qtsim import Grid, EGPESolver, bare_dipolar_symbol, gamma_tilde, Q5

eps_dd, n0as3 = 1.4, 5e-5
run = archive.new_run('dipolar groundstate',
                      params={'eps_dd': eps_dd, 'n0_as3': n0as3, 'N': 128})

Lbox = 32.0
grid = Grid((128, 128), (Lbox, Lbox))
gam = gamma_tilde(eps_dd, n0as3)
# 2D: dipoles polarized IN-PLANE.  Truncating a 3D axis (0,0,1) to 2D
# gives (0,0) -- the zero vector -- and NaN everywhere.  Guarded now.
Dk = bare_dipolar_symbol(grid, (0, 1))
print('eps_dd = %.2f   Q5 = %.4f   gamma_tilde = %.6f'
      % (eps_dd, Q5(eps_dd), gam))

rng = np.random.default_rng(42)
s = EGPESolver(grid, eps_dd=eps_dd, gamma=gam, Dk=Dk)
s.psi = (np.ones(grid.shape, dtype=complex)
         + 0.05*(rng.standard_normal(grid.shape)
                 + 1j*rng.standard_normal(grid.shape)))
Nt = float(grid.integrate(np.ones(grid.shape)))
s.psi *= np.sqrt(Nt / s.norm())

print('relaxing in imaginary time ...')
s.step_imag(0.005, 4000, norm_target=Nt)
s.step_imag(0.001, 4000, norm_target=Nt)
mu, res = s.mu_and_residual()
print('mu = %.5f   stationarity residual = %.2e' % (mu, res))

n = np.abs(s.psi)**2
contrast = float((n.max() - n.min()) / n.mean())

# dominant modulation wavevector (ignore the k=0 mean)
spec = np.abs(np.fft.fftn(n - n.mean()))
spec[0, 0] = 0.0
i0, j0 = np.unravel_index(np.argmax(spec), spec.shape)
kmag = float(np.hypot(grid.K[0][i0, j0], grid.K[1][i0, j0]))
lam = 2*np.pi/kmag if kmag > 0 else float('inf')
periods = Lbox/lam if np.isfinite(lam) else 0.0

print()
print('density contrast      = %.4f' % contrast)
print('dominant wavelength   = %.2f xi' % lam)
print('periods across box    = %.2f' % periods)

if contrast < 0.05:
    verdict = 'UNIFORM (no modulation)'
elif periods < 2.0:
    verdict = 'BOX-SCALE ARTIFACT (not a crystal)'
else:
    verdict = 'MODULATED, %.1f periods (crystal-like)' % periods
print('verdict: %s' % verdict)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(n.T, origin='lower',
               extent=[-Lbox/2, Lbox/2, -Lbox/2, Lbox/2], cmap='viridis')
plt.colorbar(im, label='density n')
ax.set_title('eps_dd = %.2f\n%s' % (eps_dd, verdict), fontsize=10)
ax.set_xlabel('x / xi'); ax.set_ylabel('y / xi')
plt.tight_layout()
run.save_figure(fig, 'groundstate-density')
plt.show()

run.save_checkpoint(s.psi, step=8000, t=0.0)
run.save_diagnostics([{'eps_dd': eps_dd, 'mu': mu, 'residual': res,
                       'contrast': contrast, 'wavelength_xi': lam,
                       'periods_across_box': periods,
                       'verdict': verdict}])
run.finish('completed')

print()
print('WHY THIS IS NOT YET A SUPERSOLID:')
print('  The 2D kernel here is the bare periodic dipolar symbol, not')
print('  the quasi-2D projected kernel that gives a real roton minimum')
print('  and a droplet lattice at a physical wavelength.')
print('  Implementing that kernel and reproducing the published 164Dy')
print('  lattice constant is validation rung V4 -- the next task in')
print('  ROADMAP.md.  Until then this cell demonstrates machinery,')
print('  not supersolid physics.')

## 5. Campaign run

In [ ]:
# CELL 5 - Short campaign run through the archive.
import subprocess, sys

cmd = [sys.executable, 'campaign/template_scan.py',
       '--eps_dd', '1.3', '--Ma', '0.5', '--seed', '0',
       '--N', '128', '--L', '64',
       '--T_relax', '20', '--T_stir', '20', '--T_decay', '40',
       '--dt', '0.01',
       '--checkpoint_every', '400', '--use_archive']
r = subprocess.run(cmd, cwd=DEST, capture_output=True, text=True)
print(r.stdout[-3000:])
if r.stderr.strip():
    print('STDERR:', r.stderr[-1500:])

## 6. Archive contents

In [ ]:
# CELL 6 - Everything saved in Drive, run by run.
import os

print('=' * 62)
archive.list_runs()
print('=' * 62)

for r in archive.load_index().get('runs', []):
    d = os.path.join(archive.project_dir, r['run_id'])
    print()
    print(r['run_id'])
    print('   title   :', r.get('title', '?'))
    print('   started :', r.get('started', '?'))
    print('   status  :', r.get('status', '?'))
    if r.get('parameters'):
        print('   params  :', r['parameters'])
    for sub in ('checkpoints', 'diagnostics', 'figures', 'logs'):
        p = os.path.join(d, sub)
        if not os.path.isdir(p):
            continue
        for f in sorted(os.listdir(p)):
            kb = os.path.getsize(os.path.join(p, f)) / 1024.0
            print('     %s/%s  (%.1f KB)' % (sub, f, kb))

print()
print('Drive location: MyDrive/Research/Quantum_Turbulence/')